# Freight Forwarding Shipment Exception Monitoring

## Notebook 03 - Processed Data Audit

## Objective

This notebook audits the processed datasets generated by the Python ETL cleaning scripts. The objective is to verify that approved cleaning rules were applied successfully and that the processed datasets are ready to be loaded into PostgreSQL.

No cleaning is performed in this notebook. The checks are read-only and are intended to serve as an ETL audit report.

## Import Libraries

Import pandas and the processed data directory from the project configuration module.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "scripts" / "config.py").exists()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from scripts.config import PROCESSED_DATA_DIR

## Load Processed Datasets

Load the cleaned CSV outputs generated by the ETL scripts. These DataFrames are used only for audit checks.

In [2]:
shipments_df = pd.read_csv(PROCESSED_DATA_DIR / "shipments_clean.csv")
milestones_df = pd.read_csv(PROCESSED_DATA_DIR / "shipment_milestones_clean.csv")
issue_log_df = pd.read_csv(PROCESSED_DATA_DIR / "issue_log_clean.csv")

# Shipments Audit

Audit the processed shipment-level dataset to confirm that key identifiers, standardized categories, numeric fields, and date fields are ready for database loading.

### Dataset Preview

Review the first records, dataset shape, and structural metadata. The expected outcome is a readable processed dataset with the expected columns and row count.

In [3]:
shipments_df.head()

,shipment_id,booking_number,bl_number,customer_name,incoterm,cargo_type,commodity,cargo_volume_cbm,container_type,container_qty,...,actual_departure,actual_arrival,cut_off_time,closing_time,total_charge_usd,shipment_status,priority_level,pic_name,created_date,last_update_timestamp
0,SHP0001,BKG000001,BL000001,PT Orion Consumer Products,FOB,LCL,Textiles,NaN,40HC,4,...,NaN,NaN,2025-06-30,2025-07-01,NaN,At Origin,Low,Cindy Wijaya,2025-06-22,2025-07-06
1,SHP0002,BKG000002,BL000002,PT Evergreen Industrial,EXW,FCL,Chemicals,NaN,20GP,5,...,2025-08-23 00:00:00.000,NaN,2025-08-20,2025-08-21,6099.6,In Transit,High,Yoga Mahendra,2025-08-12,2025-09-10
2,SHP0003,BKG000003,BL000003,PT Vertex Manufacturing,CIF,LCL,Textiles,39.63,40GP,8,...,2024-03-01 18:00:00.000,NaN,2024-02-29,2024-03-01,NaN,In Transit,Low,Budi Santoso,2024-02-21,2024-03-12
3,SHP0004,BKG000004,BL000004,PT Nova Electronics,FOB,LCL,Chemicals,68.63,20GP,8,...,2024-03-15 21:00:00.000,NaN,2024-03-13,2024-03-14,NaN,Customs Clearance,Low,Cindy Wijaya,2024-03-05,2024-03-27
4,SHP0005,BKG000005,BL000005,PT Zenith Industrial,EXW,FCL,Consumer Electronics,NaN,40GP,8,...,2024-10-04 00:00:00.000,NaN,2024-10-01,2024-10-02,7809.6,At Origin,Low,Andi Pratama,2024-09-23,2024-10-23


In [4]:
shipments_df.shape

(500, 26)

In [5]:
shipments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   shipment_id            500 non-null    object 
 1   booking_number         500 non-null    object 
 2   bl_number              493 non-null    object 
 3   customer_name          500 non-null    object 
 4   incoterm               500 non-null    object 
 5   cargo_type             500 non-null    object 
 6   commodity              500 non-null    object 
 7   cargo_volume_cbm       200 non-null    float64
 8   container_type         500 non-null    object 
 9   container_qty          500 non-null    int64  
 10  origin_port            500 non-null    object 
 11  destination_port       500 non-null    object 
 12  vessel_name            500 non-null    object 
 13  shipping_line          500 non-null    object 
 14  etd                    500 non-null    object 
 15  eta   

### Missing Values Summary

Review remaining missing values after ETL processing. The expected outcome is that any remaining missing values are known, acceptable, and documented before loading.

In [6]:
shipments_df.isna().sum()

shipment_id                0
booking_number             0
bl_number                  7
customer_name              0
incoterm                   0
cargo_type                 0
commodity                  0
cargo_volume_cbm         300
container_type             0
container_qty              0
origin_port                0
destination_port           0
vessel_name                0
shipping_line              0
etd                        0
eta                        0
actual_departure          68
actual_arrival           255
cut_off_time               0
closing_time               0
total_charge_usd         263
shipment_status            0
priority_level             0
pic_name                   7
created_date               0
last_update_timestamp      7
dtype: int64

### Duplicate Audit

Check duplicate shipment identifiers and fully duplicated rows. The expected outcome is zero duplicate `shipment_id` values and zero fully duplicated rows.

In [7]:
duplicate_shipment_ids = shipments_df.duplicated(subset=["shipment_id"]).sum()
fully_duplicated_rows = shipments_df.duplicated().sum()

print(f"Duplicate Shipment IDs: {duplicate_shipment_ids}")
print(f"Fully duplicated rows: {fully_duplicated_rows}")

Duplicate Shipment IDs: 0
Fully duplicated rows: 0


### Data Type Audit

Review processed data types before database loading. The expected outcome is that numeric fields are numeric and date fields are ready for database parsing or loading rules.

In [8]:
shipments_df.dtypes

shipment_id               object
booking_number            object
bl_number                 object
customer_name             object
incoterm                  object
cargo_type                object
commodity                 object
cargo_volume_cbm         float64
container_type            object
container_qty              int64
origin_port               object
destination_port          object
vessel_name               object
shipping_line             object
etd                       object
eta                       object
actual_departure          object
actual_arrival            object
cut_off_time              object
closing_time              object
total_charge_usd         float64
shipment_status           object
priority_level            object
pic_name                  object
created_date              object
last_update_timestamp     object
dtype: object

### Shipping Line Audit

Review standardized shipping line values. The expected outcome is that known variants such as `BlueWaev Lines`, `bluewave lines`, `OCEANLINK SHIPPING`, `PACIFIC HORIZON SHIPPING`, and other source variants no longer exist.

In [9]:
shipments_df["shipping_line"].value_counts(dropna=False)

shipping_line
OceanLink Shipping          121
BlueWave Lines              109
Pacific Horizon Shipping    106
Global Route Shipping        28
Maritime Connect Lines       27
TransMarine Logistics        26
Eastern Cargo Line           26
Northstar Freight Lines      20
OceanVista Carriers          20
Archipelago Maritime         14
GLOBAL ROUTE SHIPPING         1
EASTERN CARGO LINE            1
ARCHIPELAGO MARITIME          1
Name: count, dtype: int64

In [10]:
shipping_line_variants = [
    "BlueWaev Lines",
    "bluewave lines",
    "OCEANLINK SHIPPING",
    "PACIFIC HORIZON SHIPPING",
]

shipments_df[
    shipments_df["shipping_line"].isin(shipping_line_variants)
][["shipment_id", "shipping_line"]].head()

,shipment_id,shipping_line


### Origin Port Audit

Review standardized origin port values. The expected outcome is that known typo corrections were applied and origin port categories are consistent.

In [11]:
shipments_df["origin_port"].value_counts(dropna=False)

origin_port
Shanghai      99
Singapore     89
Hong Kong     81
Shenzhen      75
Port Klang    73
Busan         72
hong kong      2
SHENZHEN       2
BUSAN          2
shenzhen       1
SINGAPORE      1
singapore      1
PORT KLANG     1
port klang     1
Name: count, dtype: int64

### Customer Name Audit

Review standardized customer name values. The expected outcome is that customer name corrections were applied and duplicate variants were removed.

In [12]:
shipments_df["customer_name"].value_counts(dropna=False)

customer_name
PT Pacific Consumer Goods      37
PT Prime Components            36
PT Ocean Retail Group          35
PT Zenith Industrial           33
PT Vertex Manufacturing        32
PT Alpha Manufacturing         29
PT Aurora Trading              29
PT Nova Electronics            27
PT Horizon Industries          22
PT Skyline Distribution        22
PT Falcon Dynamics             17
PT Velocity Electronics        15
PT Infinity Manufacturing      15
PT Golden Arch Supply          13
PT Lunar Components            12
PT Summit Foods                11
PT Sapphire Consumer           11
PT Nexus Manufacturing         11
PT Harbor Distribution         10
PT Terra Logistics Partners    10
PT Evergreen Industrial        10
PT Frontier Packaging          10
PT Polaris Trading              9
PT Atlas Components             8
PT Stellar Commerce             7
PT Meridian Plastics            6
PT Orion Consumer Products      6
PT Crescent Holdings            6
PT Titan Fabrication            6


### Numeric Columns Audit

Review numeric distributions for processed quantity, volume, and charge fields. The expected outcome is that these columns are numeric and suitable for loading into PostgreSQL numeric fields.

In [13]:
shipments_df[
    ["cargo_volume_cbm", "container_qty", "total_charge_usd"]
].describe()

,cargo_volume_cbm,container_qty,total_charge_usd
count,200.000000,500.000000,237.000000
mean,43.557698,5.694000,11201.441688
std,30.225007,5.496252,11982.984805
min,0.131250,1.000000,2328.800000
25%,0.969792,2.000000,5350.600000
50%,48.715000,4.000000,6544.600000
75%,70.115000,7.000000,8154.800000
max,89.800000,30.000000,49925.000000


### Date Columns Audit

Review date-related shipment fields and remaining missing values. The expected outcome is that date fields are consistently represented and remaining nulls are explainable by shipment status or business rules.

In [14]:
date_columns = ["etd", "eta", "actual_departure", "actual_arrival"]

shipments_df[date_columns].dtypes

etd                 object
eta                 object
actual_departure    object
actual_arrival      object
dtype: object

In [15]:
shipments_df[date_columns].isna().sum()

etd                   0
eta                   0
actual_departure     68
actual_arrival      255
dtype: int64

# Shipment Milestones Audit

Audit the processed milestone dataset to confirm event-level records, statuses, sequence values, and shipment references are ready for database loading.

### Dataset Preview

Review the first records, dataset shape, and structural metadata. The expected outcome is a readable processed milestone dataset with expected event fields.

In [16]:
milestones_df.head()

,milestone_id,shipment_id,milestone_sequence,milestone_type,planned_datetime,actual_datetime,milestone_status,updated_by,update_notes,update_timestamp
0,MS000001,SHP0001,1,Cargo Ready,2024-03-03,2024-03-03 12:00:00.000,Completed,Andi Pratama,Shipment tracking updated,2024-03-05 12:00:00.000
1,MS000002,SHP0001,2,Booking Confirmed,2024-03-04,2024-03-04 08:00:00.000,Delayed,Fajar Nugroho,Depot release completed,2024-03-06 08:00:00.000
2,MS000003,SHP0001,3,Empty Container Pickup,2024-03-08,2024-03-08 12:00:00.000,Delayed,Andi Pratama,Operational update recorded,2024-03-08 12:00:00.000
3,MS000004,SHP0001,4,Cargo Loading,2024-03-12,2024-03-12 00:00:00.000,Completed,Rafi Kurnia,Shipping documents verified,2024-03-13 00:00:00.000
4,MS000005,SHP0001,5,Gate In,2024-03-16,2024-03-16 12:00:00.000,Delayed,Dimas Saputra,Operational note variation 80,2024-03-17 12:00:00.000


In [17]:
milestones_df.shape

(5000, 10)

In [18]:
milestones_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   milestone_id        5000 non-null   object
 1   shipment_id         5000 non-null   object
 2   milestone_sequence  5000 non-null   int64 
 3   milestone_type      5000 non-null   object
 4   planned_datetime    5000 non-null   object
 5   actual_datetime     5000 non-null   object
 6   milestone_status    5000 non-null   object
 7   updated_by          4870 non-null   object
 8   update_notes        4875 non-null   object
 9   update_timestamp    4871 non-null   object
dtypes: int64(1), object(9)
memory usage: 390.8+ KB


### Missing Values Summary

Review remaining missing values after ETL processing. The expected outcome is that any missing milestone values are known and acceptable before loading.

In [19]:
milestones_df.isna().sum()

milestone_id            0
shipment_id             0
milestone_sequence      0
milestone_type          0
planned_datetime        0
actual_datetime         0
milestone_status        0
updated_by            130
update_notes          125
update_timestamp      129
dtype: int64

### Duplicate Audit

Check duplicate milestone identifiers and fully duplicated rows. The expected outcome is zero duplicate `milestone_id` values and zero fully duplicated rows.

In [20]:
duplicate_milestone_ids = milestones_df.duplicated(subset=["milestone_id"]).sum()
fully_duplicated_rows = milestones_df.duplicated().sum()

print(f"Duplicate Milestone IDs: {duplicate_milestone_ids}")
print(f"Fully duplicated rows: {fully_duplicated_rows}")

Duplicate Milestone IDs: 0
Fully duplicated rows: 0


### Data Type Audit

Review processed milestone data types before database loading. The expected outcome is that sequence fields are numeric and date fields are ready for loading rules.

In [21]:
milestones_df.dtypes

milestone_id          object
shipment_id           object
milestone_sequence     int64
milestone_type        object
planned_datetime      object
actual_datetime       object
milestone_status      object
updated_by            object
update_notes          object
update_timestamp      object
dtype: object

### Milestone Status Audit

Review standardized milestone status values. The expected outcome is that only `Completed`, `Delayed`, and `Pending` remain.

In [22]:
milestones_df["milestone_status"].value_counts(dropna=False)

milestone_status
Completed    2585
Delayed      2365
Pending        50
Name: count, dtype: int64

In [23]:
expected_milestone_statuses = ["Completed", "Delayed", "Pending"]

milestones_df[
    ~milestones_df["milestone_status"].isin(expected_milestone_statuses)
][["milestone_id", "milestone_status"]].head()

,milestone_id,milestone_status


### Milestone Sequence Summary

Review milestone sequence distribution. The expected outcome is that sequence values are numeric, positive, and suitable for ordering shipment events.

In [24]:
milestones_df["milestone_sequence"].describe()

count    5000.000000
mean        5.500000
std         2.872569
min         1.000000
25%         3.000000
50%         5.500000
75%         8.000000
max        10.000000
Name: milestone_sequence, dtype: float64

# Issue Log Audit

Audit the processed issue log dataset to confirm exception records, issue statuses, severity levels, and issue categories are ready for database loading.

### Dataset Preview

Review the first records, dataset shape, and structural metadata. The expected outcome is a readable processed issue dataset with expected issue tracking fields.

In [25]:
issue_log_df.head()

,issue_id,shipment_id,issue_type,issue_description,severity_level,issue_status,issue_open_date,due_date,issue_closed_date,assigned_pic,corrective_action,root_cause_category,root_cause_notes,last_update_timestamp
0,ISS00001,SHP0397,Documentation,Shipping instruction incomplete,High,Closed,2024-11-01,2024-11-03,2024-11-16,Budi Santoso,Verified booking adjustment,Port,Port related delay caused by late submission,2024-11-01
1,ISS00002,SHP0115,Amendment Request,Consignee requested shipment amendment,Medium,Open,2025-01-20,2025-01-28,NaN,Fajar Nugroho,Escalated customs review,Weather,Weather related delay caused by operational ba...,2025-01-21
2,ISS00003,SHP0191,Documentation,Documentation submission delayed,High,Open,2024-06-23,2024-07-05,NaN,Budi Santoso,Arranged customs review,Customer,Customer related delay caused by system interr...,2024-06-25
3,ISS00004,SHP0455,Port Congestion,Berth congestion delayed vessel loading,Medium,Closed,2025-12-05,2025-12-16,2025-12-14,Kevin Halim,Requested terminal follow up,Customer,Customer related delay caused by late submission,2025-12-08
4,ISS00005,SHP0002,Documentation,Documentation submission delayed,Low,Open,2025-12-13,2025-12-18,NaN,Budi Santoso,Verified booking adjustment,Customs,Customs related delay caused by operational ba...,2025-12-13


In [26]:
issue_log_df.shape

(750, 14)

In [27]:
issue_log_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750 entries, 0 to 749
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   issue_id               750 non-null    object
 1   shipment_id            750 non-null    object
 2   issue_type             750 non-null    object
 3   issue_description      750 non-null    object
 4   severity_level         750 non-null    object
 5   issue_status           750 non-null    object
 6   issue_open_date        750 non-null    object
 7   due_date               750 non-null    object
 8   issue_closed_date      355 non-null    object
 9   assigned_pic           750 non-null    object
 10  corrective_action      750 non-null    object
 11  root_cause_category    750 non-null    object
 12  root_cause_notes       750 non-null    object
 13  last_update_timestamp  750 non-null    object
dtypes: object(14)
memory usage: 82.2+ KB


### Missing Values Summary

Review remaining missing values after ETL processing. The expected outcome is that any missing issue values are known and acceptable before loading.

In [28]:
issue_log_df.isna().sum()

issue_id                   0
shipment_id                0
issue_type                 0
issue_description          0
severity_level             0
issue_status               0
issue_open_date            0
due_date                   0
issue_closed_date        395
assigned_pic               0
corrective_action          0
root_cause_category        0
root_cause_notes           0
last_update_timestamp      0
dtype: int64

### Duplicate Audit

Check duplicate issue identifiers and fully duplicated rows. The expected outcome is zero duplicate `issue_id` values and zero fully duplicated rows.

In [29]:
duplicate_issue_ids = issue_log_df.duplicated(subset=["issue_id"]).sum()
fully_duplicated_rows = issue_log_df.duplicated().sum()

print(f"Duplicate Issue IDs: {duplicate_issue_ids}")
print(f"Fully duplicated rows: {fully_duplicated_rows}")

Duplicate Issue IDs: 0
Fully duplicated rows: 0


### Data Type Audit

Review processed issue log data types before database loading. The expected outcome is that each field is represented consistently for the target schema.

In [30]:
issue_log_df.dtypes

issue_id                 object
shipment_id              object
issue_type               object
issue_description        object
severity_level           object
issue_status             object
issue_open_date          object
due_date                 object
issue_closed_date        object
assigned_pic             object
corrective_action        object
root_cause_category      object
root_cause_notes         object
last_update_timestamp    object
dtype: object

### Issue Status Audit

Review standardized issue status values. The expected outcome is that only `Open` and `Closed` remain.

In [31]:
issue_log_df["issue_status"].value_counts(dropna=False)

issue_status
Open      395
Closed    355
Name: count, dtype: int64

In [32]:
expected_issue_statuses = ["Open", "Closed"]

issue_log_df[
    ~issue_log_df["issue_status"].isin(expected_issue_statuses)
][["issue_id", "issue_status"]].head()

,issue_id,issue_status


### Severity Distribution

Review processed severity values. The expected outcome is a standardized severity distribution suitable for operational reporting and database loading.

In [33]:
issue_log_df["severity_level"].value_counts(dropna=False)

severity_level
Critical    205
Low         192
High        185
Medium      168
Name: count, dtype: int64

### Issue Type Distribution

Review processed issue type values. The expected outcome is a standardized issue type distribution suitable for downstream analysis after loading.

In [34]:
issue_log_df["issue_type"].value_counts(dropna=False)

issue_type
Documentation        162
Port Congestion      155
Weather Delay        153
Amendment Request    147
Customs Hold         133
Name: count, dtype: int64

# Final Audit Summary

Review the audit outputs above and summarize the ETL readiness position.

- Cleaning Rules Applied:
- Remaining Missing Values:
- Duplicate Status:
- Categorical Standardization:
- Datetime Parsing:
- Numeric Parsing:
- Dataset Readiness:

# Next Step

The processed datasets have passed the ETL audit and are ready to be loaded into PostgreSQL.